In [3]:
library(ggplot2)
library(data.table)
library(stringr)
theme_set(theme_bw())

In [4]:
tools = c('singlem', 'sylph')
d1 = data.table(tool = tools, sample = "SRR8648366")

In [5]:
readit = function(tool, sample){
    to_read = paste0('output_',tool,'/', tool, '/',sample,'.profile')
    return(fread(to_read))
}
d2 = d1[, readit(tool, sample)[, c("coverage", "taxonomy")], by=list(tool, sample)]
d2[, relabu := coverage / sum(coverage, na.rm = TRUE), by = list(tool, sample)]
d2[, taxonomy := gsub("; ", ";", gsub("Root; ", "", taxonomy))]

In [6]:
m = dcast(d2, sample + taxonomy ~ tool, value.var = c("relabu", "coverage"), fill = 0)
setnames(m, names(m), gsub("_", "__", names(m)))

In [13]:
format_table <- function(m) {
    ra_cols <- grep("^relabu_|_relabu$", names(m), value = TRUE)
    mtmp = m[,which(!grepl("^coverage_|_coverage$", names(m))), with = FALSE]
    mtmp[
        ,
        c(ra_cols, "taxonomy") := c(
            lapply(.SD[, ra_cols, with = FALSE], scales::label_percent()),
            list(purrr::map(strsplit(taxonomy, "; ?"), function(x) paste0(tail(x,2), collapse = "; ")))
        )
    ][]
}

In [14]:
# kingdom
format_table(
    m[grep("d__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";p__.*", "", taxonomy))][relabu__sylph > 0.01]
)

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<list>,<chr>,<chr>
SRR8648366,d__Archaea,1%,1%
SRR8648366,d__Bacteria,99%,99%


In [15]:
# phylum
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";p__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";c__.*", "", taxonomy))][relabu__sylph > 0.01]
)

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<list>,<chr>,<chr>
SRR8648366,d__Bacteria; p__Actinobacteriota,60.7%,59.160%
SRR8648366,d__Bacteria; p__Bacteroidota,9.1%,8.938%
SRR8648366,d__Bacteria; p__Firmicutes,2.1%,7.805%
SRR8648366,d__Bacteria; p__Firmicutes_A,3.1%,10.407%
SRR8648366,d__Bacteria; p__Proteobacteria,14.6%,10.440%


In [16]:
# class
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";c__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";o__.*", "", taxonomy))][relabu__sylph > 0.01]
)

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<list>,<chr>,<chr>
SRR8648366,p__Actinobacteriota; c__Actinomycetia,57.1%,58.84%
SRR8648366,p__Bacteroidota; c__Bacteroidia,8.6%,8.94%
SRR8648366,p__Firmicutes; c__Bacilli,2.1%,7.80%
SRR8648366,p__Firmicutes_A; c__Clostridia,3.1%,10.41%
SRR8648366,p__Proteobacteria; c__Gammaproteobacteria,13.3%,9.52%


In [17]:
# order
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";o__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";f__.*", "", taxonomy))][relabu__sylph > 0.01]
)

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<list>,<chr>,<chr>
SRR8648366,c__Actinomycetia; o__Actinomycetales,35.51955%,40.2508%
SRR8648366,c__Actinomycetia; o__Mycobacteriales,13.11556%,17.2483%
SRR8648366,c__Actinomycetia; o__Propionibacteriales,6.08801%,1.0256%
SRR8648366,c__Bacteroidia; o__Bacteroidales,1.49753%,7.0385%
SRR8648366,c__Bacteroidia; o__Flavobacteriales,4.13191%,1.9000%
SRR8648366,c__Bacilli; o__Lactobacillales,0.64331%,4.1448%
SRR8648366,c__Bacilli; o__Staphylococcales,0.14799%,1.2676%
SRR8648366,c__Clostridia; o__Christensenellales,0.29246%,1.3567%
SRR8648366,c__Clostridia; o__Clostridiales,0.58894%,2.2670%


In [18]:
# family
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";f__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";g__.*", "", taxonomy))][relabu__sylph > 0.01]
)

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<list>,<chr>,<chr>
SRR8648366,o__Actinomycetales; f__Brevibacteriaceae,1.17638%,4.8462%
SRR8648366,o__Actinomycetales; f__Dermatophilaceae,24.65154%,32.9502%
SRR8648366,o__Mycobacteriales; f__Mycobacteriaceae,7.82464%,16.3370%
SRR8648366,o__Propionibacteriales; f__Propionibacteriaceae,2.32079%,1.0256%
SRR8648366,o__Bacteroidales; f__Bacteroidaceae,0.26653%,1.6323%
SRR8648366,o__Bacteroidales; f__Dysgonomonadaceae,0.49255%,2.6270%
SRR8648366,o__Flavobacteriales; f__Flavobacteriaceae,2.52692%,1.6341%
SRR8648366,o__Lactobacillales; f__Lactobacillaceae,0.21871%,1.3850%
SRR8648366,o__Lactobacillales; f__Streptococcaceae,0.31058%,2.4729%


In [19]:
# How well does genus level rescue some of the missing genomes? First need to remake the table with genus level, annoying since kraken profiles are filled, when the rest aren't.
format_table(
    m[!grepl("d__Archaea", taxonomy) & grepl(";g__", taxonomy),
      lapply(.SD, sum),
      by = list(sample, taxonomy = gsub(";s__.*", "", taxonomy))][relabu__sylph > 0.01]
)

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<list>,<chr>,<chr>
SRR8648366,f__Brevibacteriaceae; g__Brevibacterium,0.933%,4.8462%
SRR8648366,f__Dermatophilaceae; g__F2B08,2.608%,13.4211%
SRR8648366,f__Dermatophilaceae; g__Ornithinimicrobium,4.644%,18.7868%
SRR8648366,f__Mycobacteriaceae; g__Corynebacterium,1.867%,12.7717%
SRR8648366,f__Mycobacteriaceae; g__Dietzia,0.579%,1.0377%
SRR8648366,f__Mycobacteriaceae; g__Mycobacterium,2.122%,2.1288%
SRR8648366,f__Bacteroidaceae; g__Prevotella,0.194%,1.1667%
SRR8648366,f__Dysgonomonadaceae; g__Proteiniphilum,0.289%,1.4491%
SRR8648366,f__Flavobacteriaceae; g__Aequorivita,0.649%,1.5812%


In [20]:
# Species level
format_table(m[!grepl("d__Archaea", taxonomy) & grepl(";s__", taxonomy)][relabu__sylph > 0.01])

sample,taxonomy,relabu__singlem,relabu__sylph
<chr>,<list>,<chr>,<chr>
SRR8648366,g__Brevibacterium; s__Brevibacterium intestinavium,0.6838%,4.2596%
SRR8648366,g__F2B08; s__F2B08 sp012729695,0.1792%,1.1243%
SRR8648366,g__F2B08; s__F2B08 sp012838445,2.0117%,12.2968%
SRR8648366,g__Ornithinimicrobium; s__Ornithinimicrobium sp003577095,1.4253%,18.7868%
SRR8648366,g__Corynebacterium; s__Corynebacterium casei,0.1618%,1.2108%
SRR8648366,g__Corynebacterium; s__Corynebacterium humireducens,0.6176%,3.2112%
SRR8648366,g__Corynebacterium; s__Corynebacterium pollutisoli,0.2212%,1.5277%
SRR8648366,g__Corynebacterium; s__Corynebacterium sp012838715,0.0803%,1.0554%
SRR8648366,g__Corynebacterium; s__Corynebacterium sp012838985,0.3758%,5.1237%
